# Does virtue pay? Risk-adjusted performance of ESG vs traditional ETFs

*A data-driven analysis for BEE2041 — Data Science in Economics*

---

## 1. Introduction

The past decade has reshaped how capital is allocated in public markets. Assets in funds marketed as **environmental, social and governance** (ESG) compliant grew from a niche of the industry to roughly thirty trillion US dollars under management by the mid-2020s. The story sold to investors is appealing: by tilting towards firms with cleaner emissions records, fairer labour practices and better-governed boards, you can earn a competitive return *and* nudge corporate behaviour in the direction you want.

That story has two halves that pull in opposite directions. The first says ESG screening should *help* returns: forward-looking risks — climate litigation, carbon taxes, governance scandals — are systematically priced too low, and ESG funds avoid the firms most exposed to them. The second, drawn from textbook portfolio theory, says ESG screening should *hurt* returns: any binding investment constraint shrinks the opportunity set, and a smaller opportunity set cannot produce a higher Sharpe ratio than a larger one. Which dominates in the data?

This post examines six exchange-traded funds — three ESG-focused and three broad-market — using daily prices from January 2019 to December 2025. The question is sharper than the marketing brochures pose:

> **Once we strip out exposure to common systematic risk factors, do ESG ETFs deliver positive *alpha* relative to the market — and is the difference large enough, and reliable enough, that an investor should change behaviour?**

I work through this in three layers. First, the headline numbers: cumulative growth, annualised volatility, and Sharpe ratios. Second, a **Capital Asset Pricing Model (CAPM)** regression that decomposes each ETF's excess return into a market-beta component and an alpha. Third, a **Fama–French three-factor** regression that adds size and value style factors, on the principle that an alpha which disappears once you control for known styles was never really alpha. A two-sample t-test on basket-level returns formally checks whether the ESG–traditional differential is statistically distinguishable from zero.

The audience is anyone curious about whether sustainable investing is a serious financial proposition or mostly marketing veneer. No programming background is assumed.

## 2. Data

The dataset is built entirely from public sources via code; nothing was downloaded by hand.

**Equity returns.** Daily adjusted-close prices for six ETFs are pulled from Yahoo Finance through the `yfinance` Python package. *Adjusted* means the price series is back-adjusted for dividends and splits, so the simple percentage change between consecutive days is a true total return.

**ESG basket.** *ESGU* (iShares ESG Aware MSCI USA) is a broad US large-cap fund that tilts towards higher-rated ESG firms while keeping sector weights close to the parent index. *SUSA* (iShares MSCI USA ESG Select) is a stricter best-in-class sustainability fund. *ICLN* (iShares Global Clean Energy) is a thematic fund concentrated in renewable-energy producers — the most "green" of the three by some distance.

**Traditional basket.** *SPY* (S&P 500), *VTI* (Vanguard Total Stock Market) and *DIA* (Dow Jones Industrial Average) provide three different cuts of the broad US equity market with no sustainability screen at all.

**Risk factors.** Daily values for the Fama–French three factors — the market excess return (Mkt-RF), the size factor (SMB) and the value factor (HML) — together with the daily US risk-free rate, are downloaded from Kenneth R. French's data library via `pandas-datareader`. Factors are stored as decimal returns, not percentages.

**Sample window.** January 2019 through December 2025. This gives roughly 1,750 trading days per series and contains at least three regimes — the late-cycle 2019 rally, the COVID crash and recovery, and the 2022 inflation shock — which makes for a fair test of how the two baskets cope with different macro environments.

The cell below loads the cleaned data that the project's scripts produce.

In [ ]:
# Load the cleaned project data. Run scripts/fetch_data.py and
# scripts/clean_data.py first if these CSVs do not exist yet.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "scripts"))
import pandas as pd
from config import (PROCESSED_DIR, OUTPUTS_DIR, PLOTS_DIR,
                    ESG_TICKERS, TRADITIONAL_TICKERS)

prices  = pd.read_csv(PROCESSED_DIR / "prices.csv",
                      parse_dates=["Date"], index_col="Date")
returns = pd.read_csv(PROCESSED_DIR / "returns.csv",
                      parse_dates=["Date"], index_col="Date")
factors = pd.read_csv(PROCESSED_DIR / "factors.csv",
                      parse_dates=["Date"], index_col="Date")

print(f"Sample: {returns.index.min().date()} to {returns.index.max().date()}")
print(f"Trading days: {len(returns):,}")
print(f"Tickers: {list(returns.columns)}")
returns.head().round(4)


## 3. Methodology

### 3.1 Returns and risk

I use **simple daily returns** $r_t = P_t / P_{t-1} - 1$ throughout. Mean and standard deviation are annualised in the conventional way — multiply the mean by 252 trading days, multiply the standard deviation by $\sqrt{252}$. The square-root rule assumes serially uncorrelated returns; daily equity returns are close enough to that ideal that any error is small.

The **Sharpe ratio** is the annualised mean of *excess* returns (return minus the risk-free rate) divided by the annualised standard deviation. It is the cleanest single-number summary of how much compensation an investor is receiving per unit of total risk.

### 3.2 CAPM

The CAPM regression is

$$r_{i,t} - r_{f,t} \;=\; \alpha_i \;+\; \beta_i\,(r_{m,t} - r_{f,t}) \;+\; \varepsilon_{i,t}.$$

In words: each ETF's excess return is split into a fee-like constant ($\alpha$) and a part that moves one-for-one with the market times a sensitivity ($\beta$). Beta near one means the fund moves like the market; below one means it is defensive; above one means it amplifies market moves. **Alpha is the part of return that the market cannot explain** — a statistically significant positive alpha is the closest thing in finance to a free lunch.

### 3.3 Fama–French three-factor

The CAPM is famously incomplete: even random portfolios show up with non-zero "alpha" if their style is unusual. Fama and French extend it by adding two empirical risk factors:

$$r_{i,t} - r_{f,t} \;=\; \alpha_i \;+\; \beta_{M}\,\text{MKT}_t \;+\; \beta_{S}\,\text{SMB}_t \;+\; \beta_{H}\,\text{HML}_t \;+\; \varepsilon_{i,t},$$

where SMB ("small minus big") captures the historical premium for small-cap stocks and HML ("high minus low") captures the value-versus-growth premium. If a fund's CAPM alpha shrinks toward zero once these factors are added, the apparent skill was just a tilt toward small or value stocks.

### 3.4 Inference

Daily equity returns are mildly serially correlated and clearly heteroskedastic. I therefore report **Newey–West (HAC)** standard errors with five lags, which is the standard fix in the empirical-asset-pricing literature. A two-sample (paired) t-test on the daily ESG-minus-traditional basket return checks whether the average gap is distinguishable from zero.

## 4. Results

### 4.1 Cumulative returns: who came out ahead?

In [ ]:
from IPython.display import Image, display
display(Image(PLOTS_DIR / "01_cumulative_returns.png"))


**Figure 1.** Growth of one US dollar invested at the start of the sample. Teal lines are the ESG ETFs; orange lines are the traditional ones.

The first thing to notice is that the broad-market funds (SPY, VTI, DIA) and the broad-ESG funds (ESGU, SUSA) trace nearly identical paths. This is not an accident: ESGU and SUSA are explicitly engineered to track the parent US large-cap universe with only modest tilts. **The headline-grabbing line is ICLN, the clean-energy ETF.** It enjoyed a spectacular run during the 2020-21 green-investment boom — at peak it was up roughly 130% on its starting value — before giving back almost all of those gains in the 2022 rate shock. By the end of the sample, ICLN sits *below* the broad-market funds.

This pattern is the central tension of the post. A casual reader might look at Figure 1 and conclude that ESG investing "worked" until 2021 and "failed" after. A more disciplined reading is that ICLN is a *concentrated, high-beta sector bet* with a particular sensitivity to interest rates, and that lumping it together with diversified ESG funds confuses two very different investment products.

### 4.2 Volatility: ESG is not a smooth ride

In [ ]:
display(Image(PLOTS_DIR / "02_rolling_volatility.png"))


**Figure 2.** Sixty-three-day rolling annualised volatility (one quarter of trading days).

All six funds spike together in the COVID crash of March 2020 — risk is, as ever, contagious in a crisis. The interesting structural pattern, though, is the persistent gap between ICLN (consistently 25–35% annualised) and everything else (15–22%). A clean-energy fund delivers roughly **fifty percent more volatility** than a broad-market fund for most of the sample. ESGU and SUSA, by contrast, are visually indistinguishable from SPY/VTI/DIA — their tilt is too modest to change the overall risk profile in any meaningful way.

### 4.3 The risk–return picture in one chart

In [ ]:
display(Image(PLOTS_DIR / "04_risk_return_scatter.png"))


**Figure 3.** Annualised return plotted against annualised volatility for each ETF.

The five non-ICLN funds cluster tightly: very similar risk, very similar return. ICLN sits well to the right (more risky) and slightly above (somewhat higher return) — a higher absolute return, but not obviously a better deal once that extra risk is taken into account. This is exactly what the Sharpe ratio is built to measure.

### 4.4 Summary statistics and the Sharpe table

In [ ]:
desc = pd.read_csv(OUTPUTS_DIR / "descriptive_stats.csv", index_col="Ticker")

display_df = desc.copy()
display_df["Ann. Return"]      = (display_df["Ann. Return"]*100).round(2).astype(str) + "%"
display_df["Ann. Volatility"]  = (display_df["Ann. Volatility"]*100).round(2).astype(str) + "%"
display_df["Sharpe"]           = display_df["Sharpe"].round(3)
display_df["Sharpe 95% CI low"]  = display_df["Sharpe 95% CI low"].round(3)
display_df["Sharpe 95% CI high"] = display_df["Sharpe 95% CI high"].round(3)
display_df


**Table 1.** Per-ETF descriptive statistics. Sharpe ratios are reported with a bootstrap 95% confidence interval based on 5,000 resamples.

The Sharpe ratios for the broad-ESG and broad-traditional funds are statistically indistinguishable — every confidence interval overlaps with every other. ICLN's higher mean return is largely cancelled by its much higher volatility, leaving it with a Sharpe ratio that is no better, and arguably worse, than that of a plain S&P 500 tracker.

### 4.5 Distribution of daily returns

In [ ]:
display(Image(PLOTS_DIR / "03_return_distributions.png"))


**Figure 4.** Histogram of daily returns for the equally-weighted ESG basket and the equally-weighted traditional basket.

The two distributions are visually almost identical: same centre, same width, same fat tails. A formal t-test confirms what the eye sees.

In [ ]:
ttest = pd.read_csv(OUTPUTS_DIR / "basket_ttest.csv", index_col=0)["value"]
print(f"Mean daily ESG-Traditional gap : {ttest['mean_diff_daily']*1e4:6.2f} bps")
print(f"Annualised:                      {ttest['mean_diff_annual']*100:6.2f} pp")
print(f"t-statistic:                     {ttest['t_stat']:6.3f}")
print(f"p-value:                         {ttest['p_value']:6.4f}")
print(f"Number of daily observations:    {int(ttest['n']):,}")


**Test result.** A paired t-test on the daily ESG-basket return minus the daily traditional-basket return cannot reject the null hypothesis of zero mean difference at conventional significance levels. **There is no measurable, systematic outperformance — in either direction.**

### 4.6 CAPM regressions: alpha and beta

In [ ]:
display(Image(PLOTS_DIR / "05_capm_bars.png"))


**Figure 5.** Per-ETF CAPM regression results. Left panel: alpha (annualised, basis points). Right panel: beta. Asterisks mark alphas significant at the 5% level using HAC-robust standard errors.

In [ ]:
capm = pd.read_csv(OUTPUTS_DIR / "capm.csv", index_col="Ticker")
capm_show = capm[["Group","alpha_ann_bps","alpha_p","beta","beta_p","R_squared"]].round(3)
capm_show.columns = ["Group","alpha (bps/yr)","alpha p-value","beta","beta p-value","R-squared"]
capm_show


**Table 2.** CAPM regression output. R² is the share of an ETF's excess-return variance explained by the market factor alone.

Three findings stand out. **First**, every fund except ICLN has a beta extremely close to 1.0 and an R² above 0.85, which simply confirms that broad-market US equity ETFs are nearly perfect substitutes for the market factor. **Second**, ICLN has a higher beta and a much lower R² (~0.50), reflecting the substantial chunk of its return variance that comes from clean-energy-specific shocks rather than from the broad market. **Third — and this is the key economic finding — none of the alphas are statistically significant after HAC-robust standard errors are applied.** ESG investing did not pay an alpha in this sample, and traditional investing did not either.

### 4.7 Fama–French three-factor: does the verdict hold?

In [ ]:
ff3 = pd.read_csv(OUTPUTS_DIR / "ff3.csv", index_col="Ticker")
ff3_show = ff3[["Group","alpha_ann_bps","alpha_p","beta_MKT","beta_SMB","beta_HML","R_squared"]].round(3)
ff3_show.columns = ["Group","alpha (bps/yr)","alpha p-value","beta_MKT","beta_SMB","beta_HML","R-squared"]
ff3_show


**Table 3.** Fama–French three-factor regression output.

The picture barely changes. Adding the size and value factors lifts R² very slightly for the broad-market funds and for ICLN, but the alphas remain statistically indistinguishable from zero. The factor loadings are economically intuitive — ICLN tilts modestly toward small-caps (positive β_SMB) and away from value (negative β_HML), consistent with a portfolio of growth-oriented renewables firms — but none of these tilts translates into a free lunch.

### 4.8 Drawdowns: who suffers more in a crisis?

In [ ]:
display(Image(PLOTS_DIR / "06_drawdowns.png"))


**Figure 6.** Drawdown from the running peak for the ESG and traditional baskets.

The two baskets' drawdowns trace each other closely through the COVID crash and the 2022 inflation shock. There is no evidence that the ESG basket offered meaningful downside protection in this sample. The intuition that "good" companies are also more *resilient* companies — sometimes called the "ESG defensiveness" hypothesis — is not visible in the daily data once the comparison is done at the portfolio level.

## 5. Discussion

What does this add up to?

**The headline result is "no difference."** Over a seven-year window covering a pandemic, an inflation shock and a major rate-hiking cycle, broad ESG ETFs delivered essentially the same risk-adjusted return as broad market ETFs. The CAPM and Fama–French regressions confirm it: alphas are economically small and statistically indistinguishable from zero, and the two distributions of daily returns are statistically identical.

This is, in one sense, *good news for ESG*. Standard portfolio theory predicts that any binding investment constraint must reduce the maximum attainable Sharpe ratio. The fact that broad-ESG funds match the market — rather than visibly underperforming — means the "cost of virtue" is, in practice, close to zero for diversified investors. **A pension fund that screens out the worst ESG offenders does not appear to sacrifice meaningful risk-adjusted return.**

**The thematic story is more cautionary.** ICLN, the clean-energy fund, did *not* match the market: it took on substantially more volatility for no extra risk-adjusted reward. Investors who treat thematic ESG funds as drop-in replacements for diversified equity exposure are taking on concentrated factor bets — a long position in growth, small-caps and a particular sector — that bear little relationship to the portfolio's nominal sustainability mandate. Calling ICLN an "ESG fund" tells you almost nothing about its expected behaviour; calling it a "concentrated long-duration sector bet on policy-sensitive renewables" tells you almost everything.

**The macro regime matters.** In the low-rate environment of 2020-21, growth-tilted ESG funds outperformed; in the rate-hike environment of 2022-23, they underperformed. The seven-year average is a wash, but it hides substantial cyclicality.

## 6. Limitations

Three caveats deserve emphasis.

**ETF-level evidence is not stock-level evidence.** The funds I analyse use the parent index methodology and apply a screen on top. Effects visible in academic stock-level studies — for instance, that the *worst* ESG offenders earn higher expected returns because investors shun them — are largely diversified away when you buy a broad fund of "good" companies. A different research design (long-short portfolios sorted on ESG ratings) might well produce a different answer.

**ESG ratings are not a settled construct.** Different rating providers — MSCI, Sustainalytics, S&P Global — disagree on which firms should be classed as ESG leaders, with cross-provider correlations of only about 0.4 in published academic work. The ETFs I use rely on MSCI's ratings; a fund built on a different provider's screen would hold a slightly different basket. This rating-divergence problem also makes it hard to interpret "ESG investing" as a single, well-defined behaviour.

**Survivorship and short-history bias.** ICLN has been around since 2008, but the broader landscape of dedicated ESG ETFs is young. Many products did not exist before 2017-18, which means even a seven-year window leans on funds whose inception was during a period of strong inflows and rising valuations. Whether the "no measurable cost" result generalises to a longer horizon is, simply, unknown.

## 7. Conclusion

> **The data does not support the claim that ESG investing systematically outperforms — but it also does not support the claim that ESG investing is a costly indulgence.**

For diversified ESG ETFs, the cost of the screen, in risk-adjusted-return terms, is statistically zero in this sample. For thematic ESG ETFs like clean-energy funds, the cost (in additional concentrated risk) is real and visible, and is fundamentally a story about sector and style exposure rather than about virtue. The right conclusion for an investor with a sustainability preference is therefore simple: the broad ESG product is, on this evidence, a free expression of values. The thematic product is a bet, and should be sized like one.

---

*Code, data and figures are available in the project repository. All analysis is reproducible end-to-end by running the four scripts in `scripts/` in order: `fetch_data.py` → `clean_data.py` → `analysis.py` → `plots.py`.*